# Text Splitters

In [38]:
from pprint import pprint

from langchain_text_splitters import RecursiveCharacterTextSplitter

text='I asked a model to read a 300‑page contract, pull out every liability clause, and rewrite the result in plain English. Within seconds it gave me a concise, accurate briefing—something that would have taken a junior associate an hour or more.That wasn’t a fluke.'
text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=20)
texts = text_splitter.split_text(text)
pprint(texts)

['I asked a model to read a 300‑page contract, pull',
 'contract, pull out every liability clause, and',
 'clause, and rewrite the result in plain English.',
 'in plain English. Within seconds it gave me a',
 'it gave me a concise, accurate briefing—something',
 'briefing—something that would have taken a junior',
 'have taken a junior associate an hour or',
 'an hour or more.That wasn’t a fluke.']


# Dcoument Loaders

In [ ]:
from anyio import Path
from langchain_community.document_loaders import TextLoader, PyPDFLoader


path = Path("post.txt")
loader = TextLoader(path, encoding="utf-8")
document = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200,    separators=[" ", ""]
)
texts = text_splitter.split_documents(document)

pprint(texts)

[Document(metadata={'source': 'post.txt'}, page_content='**LLMs in\u202f2024: The “Omni‑Capability” Year and What It Means for Builders**  \n\n---\n\n### 1. Why 2024 Feels Like a Turning Point  \n\nI asked a model to read a 300‑page contract, pull out every liability clause, and rewrite the result in plain English. Within seconds it gave me a concise, accurate briefing—something that would have taken a junior associate an hour or more.  \n\nThat wasn’t a fluke. A series of upgrades over the past year turned large language models from clever chatbots into truly *omni‑capable* assistants. In plain terms, an LLM is a massive neural network (often billions of parameters) that has been trained on a huge mix of text, images, audio, and even video. By learning to predict the next token, the model picks up the ability to write prose, reason through problems, control external tools, and now perceive the world through multiple senses.  \n\nBelow I’ll walk through the biggest shifts that happened

In [2]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFLoader

path = os.path.abspath("medical_record.pdf")
loader = PyPDFLoader(path)
documents = loader.load()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(documents)  # LangChain Document objects

store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = store.as_retriever(search_kwargs={"k": 4})

question = "i need breife summary about the patient medical record"
context = "\n\n".join(d.page_content for d in retriever.invoke(question))
print(llm.invoke(f"Using only this context:\n{context}\n\nQuestion: {question}").content)

Karim BENALI, a 55-year-old male and former smoker, has a medical record indicating Type 2 diabetes mellitus. In February 2024, he was diagnosed with hypertension and poorly controlled diabetes (HbA1c 8.1%), along with high LDL. He was prescribed lisinopril, metformin, and atorvastatin. Lisinopril was later discontinued due to an ACE-inhibitor cough and replaced with losartan. By May 2024, his HbA1c and LDL had improved.

In January 2025, he was diagnosed with mild non-proliferative diabetic retinopathy. In August 2025, he presented with exertional chest pressure, leading to a positive exercise stress test indicating inducible ischemia. Subsequently, in September 2025, he underwent coronary angiography which revealed 80% stenosis of the mid left anterior descending (LAD) artery, for which he received a percutaneous coronary intervention (PCI) with a drug-eluting stent.


# OpenDataLoaderPDFLoader

In [1]:
from langchain_opendataloader_pdf import OpenDataLoaderPDFLoader

path = "medical_record.pdf"

loader = OpenDataLoaderPDFLoader(
    file_path=[path],
    format="markdown",
    image_output="external",
    image_dir="./images",
    image_format="png",
)

documents = loader.load()

print(documents)

oct. 26, 2026 11:59:43 AM org.opendataloader.pdf.processors.DocumentProcessor preprocessing
INFO: File name: c:\Users\pc\Desktop\lang_work\Rag\medical_record.pdf
oct. 26, 2026 11:59:45 AM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Number of pages: 6
oct. 26, 2026 11:59:45 AM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Author: Synthetic dataset
oct. 26, 2026 11:59:45 AM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Title: Synthetic Patient Medical Record (RAG practice)
oct. 26, 2026 11:59:45 AM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Creation date: D:20261024170713+00'00'
oct. 26, 2026 11:59:45 AM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Modification date: D:20261024170713+00'00'
oct. 26, 2026 11:59:45 AM org.opendataloader.pdf.processors.DocumentProcessor processDocument
INFO: Processing 6 pages with 1 

# Docling

In [9]:

from pathlib import Path
import json

from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions


# 1. Define paths
pdf_path = Path("medical_record.pdf")
output_dir = Path("parsed")
images_dir = output_dir / "images"

if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found: {pdf_path.resolve()}")

output_dir.mkdir(parents=True, exist_ok=True)
images_dir.mkdir(parents=True, exist_ok=True)


# 2. Configure Docling
options = PdfPipelineOptions()
options.do_ocr = True
options.do_table_structure = True
options.generate_picture_images = True

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=options
        )
    }
)


# 3. Parse the PDF
print(f"Processing: {pdf_path.resolve()}")
result = converter.convert(pdf_path)
doc = result.document


# 4. Save Markdown
markdown = doc.export_to_markdown()

(output_dir / "document.md").write_text(
    markdown,
    encoding="utf-8",
)


# 5. Save structured JSON
doc.save_as_json(output_dir / "document.json")


# 6. Extract detected pictures and record provenance
image_inventory = []

for index, picture in enumerate(doc.pictures, start=1):
    image = picture.get_image(doc)

    provenance = picture.prov[0] if picture.prov else None
    page_number = provenance.page_no if provenance else None

    bbox = None
    if provenance and provenance.bbox:
        bbox = provenance.bbox.model_dump()

    caption = picture.caption_text(doc)

    image_path = None

    if image is not None:
        filename = f"image_{index:03}.png"
        saved_path = images_dir / filename

        # Convert to PNG when saving.
        image.save(saved_path, format="PNG")
        image_path = str(saved_path)

    image_inventory.append({
        "image_id": index,
        "page": page_number,
        "caption": caption,
        "bbox": bbox,
        "image_path": image_path,
    })

    print(
        f"Picture {index}: page={page_number}, "
        f"caption={caption or 'No caption'}, "
        f"saved={image_path is not None}"
    )


# 7. Save the image inventory
with open(
    output_dir / "image_inventory.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        image_inventory,
        file,
        ensure_ascii=False,
        indent=2,
    )


# 8. Print summary
print("\nProcessing finished.")
print(f"Pages represented: {len(doc.pages)}")
print(f"Detected pictures: {len(doc.pictures)}")
print(f"Markdown characters: {len(markdown):,}")
print(f"Markdown: {(output_dir / 'document.md').resolve()}")
print(f"JSON: {(output_dir / 'document.json').resolve()}")
print(f"Image inventory: {(output_dir / 'image_inventory.json').resolve()}")
print(f"Images folder: {images_dir.resolve()}")


c:\Users\pc\Desktop\lang_work\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing: C:\Users\pc\Desktop\lang_work\Rag\medical_record.pdf


c:\Users\pc\Desktop\lang_work\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pc\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\pc\Desktop\lang_work\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses syml


Processing finished.
Pages represented: 6
Detected pictures: 0
Markdown characters: 15,855
Markdown: C:\Users\pc\Desktop\lang_work\Rag\parsed\document.md
JSON: C:\Users\pc\Desktop\lang_work\Rag\parsed\document.json
Image inventory: C:\Users\pc\Desktop\lang_work\Rag\parsed\image_inventory.json
Images folder: C:\Users\pc\Desktop\lang_work\Rag\parsed\images


In [ ]:

from langchain_text_splitters import RecursiveCharacterTextSplitter


with open("parsed/document.md", "r", encoding="utf-8") as f:
    content = f.read()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=200)
texts = text_splitter.split_text(content)



['## Patient Medical Record\n\nLongitudinal chart, February 2024 - September 2026\n\nNotice: every name, number, date, and clinical detail in this document is invented. It exists so you can practice chunking, embedding, and retrieval (RAG) on realistic clinical text. Do not use it for medical decisions.\n\n## 1. Patient Demographics',
 '| Field                       | Value                                      |\n|-----------------------------|--------------------------------------------|\n| Full name                   | Karim BENALI                               |\n| Medical record number (MRN) | 00482917                                   |\n| Date of birth / Age         | 12 April 1971 (55 years at last visit)     |\n| Sex                         | Male                                       |\n| Occupation                  | Secondary-school mathematics teacher       |',
 '| Sex                         | Male                                       |\n| Occupation                  | Se

In [ ]:

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings


documents = [ Document(
    page_content= chunk
    ) for chunk in texts]

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store: Chroma = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./medical_db"
    )






[Document(id='194d0ea4-365f-4de6-b5aa-ebd3fe673c80', metadata={}, page_content="|   # | Question                                                               | Type          | Expected answer                                                                       |\n|-----|------------------------------------------------------------------------|---------------|---------------------------------------------------------------------------------------|\n|   1 | What is the patient's MRN?                                             | Single lookup | 00482917                                                                              |"), Document(id='ffceffb4-c7f0-4ea7-a027-4cf546f605f9', metadata={}, page_content='## Patient Medical Record\n\nLongitudinal chart, February 2024 - September 2026\n\nNotice: every name, number, date, and clinical detail in this document is invented. It exists so you can practice chunking, embedding, and retrieval (RAG) on realistic clinical text. Do not use it f

In [31]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

import os
from dotenv import load_dotenv

load_dotenv()


# -------------------------
#  Model
# -------------------------

model = ChatOpenAI(
    model="meta/muse-glimmer-30b",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0.2,
    max_tokens=500,
    reasoning_effort="medium",
)



# -------------------------
#  Create retrieval tool
# -------------------------

@tool
def search_medical_documents(query: str) -> str:
    """Search the medical documents for information relevant to the user's question."""

    docs = vector_store.similarity_search(
        query=query,
        k=4
    )

    if not docs:
        return "No relevant information was found."

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


# -------------------------
#  Create agent
# -------------------------

agent = create_agent(
    model=model,
   system_prompt = """
You are a helpful, conversational assistant that helps users understand medical records.

Your goal is to answer questions naturally, as if you were having a conversation with the user, while remaining accurate and faithful to the available medical records.

Instructions:
- Use the retrieval tool whenever you need information from the medical records to answer a question.
- Base medical-record-specific answers on the information retrieved. Never invent details or make assumptions about missing information.
- Speak naturally and directly. Answer the user's actual question first, then provide additional relevant context when it is useful.
- Adapt the length and structure of your response to the question. Give short, direct answers to simple questions and more detailed explanations when the user asks for them.
- Avoid unnecessary headings, bullet points, and formal report-style formatting in ordinary conversation. Use them when presenting complex information, multiple medications, test results, timelines, or other details that benefit from structure.
- Explain medical terminology in plain language when helpful, without changing the meaning of the records.
- If relevant information is missing or unclear, be transparent about what the records do and do not say.
- Clearly distinguish information documented in the records from general medical explanations.
- Do not make diagnoses, recommend treatments, or draw medical conclusions that are not supported by the available information.
- Maintain a calm, respectful, empathetic, and professional tone. Avoid sounding robotic, repetitive, or overly formal.
- Do not repeat the entire retrieved content when only a specific detail is needed.
""",
    tools=[search_medical_documents],
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 6),
            keep=("messages", 2),
        )
    ],
)


# -------------------------
#  Ask the agent
# -------------------------

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "what alergies the patient have?"
        }
    ]
})

print(result["messages"][-1].content)

The records list one allergy for the patient:

* **Penicillin** — diffuse skin rash with itching. Recorded as an allergy in 2009.

The record also notes a lisinopril reaction — persistent dry cough in May 2024 — but this is documented as an intolerance, not a true allergy, and is the reason lisinopril was stopped.


# MultiQueryRetriever
use llm to retrieve data usign same query meaning in diffrent versions 

In [32]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever



# Your existing Chroma vector store
base_retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

# Wrap the existing retriever with Multi-Query
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=model,
    include_original=True,
)



@tool
def search_medical_documents(query: str) -> str:
    """Search the medical documents for information relevant to the user's question."""

    docs = multi_query_retriever.invoke(
        query
    )


    if not docs:
        return "No relevant information was found."

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )



agent = create_agent(
    model=model,
    system_prompt = """
You are a helpful, conversational assistant that helps users understand medical records.

Your goal is to answer questions naturally, as if you were having a conversation with the user, while remaining accurate and faithful to the available medical records.

Instructions:
- Use the retrieval tool whenever you need information from the medical records to answer a question.
- Base medical-record-specific answers on the information retrieved. Never invent details or make assumptions about missing information.
- Speak naturally and directly. Answer the user's actual question first, then provide additional relevant context when it is useful.
- Adapt the length and structure of your response to the question. Give short, direct answers to simple questions and more detailed explanations when the user asks for them.
- Avoid unnecessary headings, bullet points, and formal report-style formatting in ordinary conversation. Use them when presenting complex information, multiple medications, test results, timelines, or other details that benefit from structure.
- Explain medical terminology in plain language when helpful, without changing the meaning of the records.
- If relevant information is missing or unclear, be transparent about what the records do and do not say.
- Clearly distinguish information documented in the records from general medical explanations.
- Do not make diagnoses, recommend treatments, or draw medical conclusions that are not supported by the available information.
- Maintain a calm, respectful, empathetic, and professional tone. Avoid sounding robotic, repetitive, or overly formal.
- Do not repeat the entire retrieved content when only a specific detail is needed.
""",
    tools=[search_medical_documents],
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 6),
            keep=("messages", 2),
        )
    ],
)


# -------------------------
#  Ask the agent
# -------------------------

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "what alergies the patient have?"
        }
    ]
})

print(result["messages"][-1].content)

The records document one allergy:

* **Penicillin** — diffuse skin rash with itching. Recorded in 2009.

The chart also notes an intolerance, not a true allergy:
* **Lisinopril, an ACE inhibitor** — persistent dry cough, recorded May 2024. It is listed as an intolerance, and lisinopril was stopped for that reason and replaced with losartan 50 mg daily.


# SelfQueryRetriever
uses metadata to filter the retrieved the data the metadata can be extracted from the query it self using SelfQueryRetriever

In [ ]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain.agents import create_agent

from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.base import AttributeInfo

load_dotenv()


# 1. Initialize the LLM
model = ChatOpenAI(
    model="meta/muse-glimmer-30b",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0.2,
    max_tokens=500,
    reasoning_effort="medium",
)


# 2. Initialize embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview"
)


# 3. Create example medical records with metadata
documents = [
    Document(
        page_content="The patient reported chest pain during exercise.",
        metadata={
            "patient_id": 101,
            "year": 2025,
            "record_type": "consultation",
        },
    ),
    Document(
        page_content="The patient had elevated glucose levels.",
        metadata={
            "patient_id": 101,
            "year": 2024,
            "record_type": "blood_test",
        },
    ),
    Document(
        page_content="The patient reported shortness of breath.",
        metadata={
            "patient_id": 202,
            "year": 2025,
            "record_type": "consultation",
        },
    ),
]


# 4. Create the Chroma vector store
# For this standalone example, create a separate collection.
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="medical_self_query_demo",
    persist_directory="./medical_self_query_db",
)


# 5. Describe the metadata fields Self-Query can use
metadata_field_info = [
    AttributeInfo(
        name="patient_id",
        description="The numeric ID of the patient",
        type="integer",
    ),
    AttributeInfo(
        name="year",
        description="The year the medical record was created",
        type="integer",
    ),
    AttributeInfo(
        name="record_type",
        description="The type of medical record, such as consultation or blood_test",
        type="string",
    ),
]


# 6. Create the Self-Query retriever
self_query_retriever = SelfQueryRetriever.from_llm(
    llm=model,
    vectorstore=vector_store,
    document_contents="Medical record text describing symptoms, consultations, and test results",
    metadata_field_info=metadata_field_info,
    search_kwargs={"k": 4},
)


# 7. Expose retrieval as a tool for the agent
@tool
def search_medical_records(query: str) -> str:
    """Search medical records by their content and metadata.
    Use this tool to find information about patients, dates,
    symptoms, consultations, and test results."""

    docs = self_query_retriever.invoke(query)

    if not docs:
        return "No matching medical records were found."

    return "\n\n".join(
        f"Patient ID: {doc.metadata.get('patient_id')}\n"
        f"Year: {doc.metadata.get('year')}\n"
        f"Record type: {doc.metadata.get('record_type')}\n"
        f"Content: {doc.page_content}"
        for doc in docs
    )


# 8. Create the agent
agent = create_agent(
    model=model,
    tools=[search_medical_records],
    system_prompt="""
You are a helpful medical-record assistant.
Use the search tool for questions about medical records.
Base your answers on the retrieved records.
Never invent missing patient information.
If the records do not provide an answer, say so clearly.
Answer naturally and directly.
""",
)


# 9. Ask a natural-language question
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What symptoms did patient 101 experience in 2025?",
        }
    ]
})

print(result["messages"][-1].content)


# ParentDocumentRetriever

The workflow is:
1. Split the original document into large parent chunks.
2. Split each parent into smaller child chunks.
3. Store parents in the document store (RAM in our example).
4. Store children and their embeddings in Chroma, along with their parent IDs.
5. When a question arrives, Chroma finds the most relevant children.
6. Use their parent IDs to retrieve the corresponding larger parents.
7. Give those parents to the LLM to generate the answer.

In [ ]:

import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

from langchain_core.documents import Document
from langchain_core.stores import InMemoryStore
from langchain_core.tools import tool

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain.agents import create_agent
from langchain.storage import LocalFileStore
from langchain.storage import create_kv_docstore


load_dotenv()

# ----------------------------------
#  Models
# ----------------------------------

model = ChatOpenAI(
    model="meta/muse-glimmer-30b",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0.2,
)

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview"
)

# ----------------------------------
#  Storage
# ----------------------------------

# Stores child chunks and their embeddings
vector_store = Chroma(
    collection_name="medical_parent_demo",
    embedding_function=embeddings,
    persist_directory="./medical_parent_db"
)

b# Persistent storage for parent documents
store = LocalFileStore("./parent_store")

docstore = create_kv_docstore(store)
# ----------------------------------
#  Splitters
# ----------------------------------

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30
)

# ----------------------------------
#  Parent Document Retriever
# ----------------------------------

retriever = ParentDocumentRetriever(
    vectorstore=vector_store,
    docstore=docstore,
    parent_splitter=parent_splitter,
    child_splitter=child_splitter,
)

# ----------------------------------
# 5. Document ingestion
# ----------------------------------

documents = [
    Document(
        page_content="""
        Patient 101 visited the clinic complaining of chest pain
        during physical exercise. The doctor recommended an ECG test.

        The patient was advised to reduce intense physical activity
        until the test results became available.

        A follow-up appointment was scheduled for two weeks later.
        During the follow-up, the patient reported improvement.
        """,
        metadata={
            "patient_id": 101,
            "year": 2025,
            "record_type": "consultation"
        }
    ),
    Document(
        page_content="""
        Patient 202 visited the clinic reporting frequent headaches.
        The doctor recommended a blood pressure examination.
        """,
        metadata={
            "patient_id": 202,
            "year": 2025,
            "record_type": "consultation"
        }
    )
]

# Automatically creates parents and children,
# stores parents in docstore and child vectors in Chroma.
retriever.add_documents(documents)

# ----------------------------------
#  Agent tool
# ----------------------------------

@tool
def search_medical_records(query: str) -> str:
    """Search medical records and return relevant parent document context."""

    # Searches child chunks, then retrieves their parents.
    docs = retriever.invoke(query)

    if not docs:
        return "No relevant medical records found."

    return "\n\n".join(
        f"Patient ID: {doc.metadata.get('patient_id')}\n"
        f"Year: {doc.metadata.get('year')}\n"
        f"Record type: {doc.metadata.get('record_type')}\n"
        f"Content: {doc.page_content}"
        for doc in docs
    )

# ----------------------------------
#  Create agent
# ----------------------------------

agent = create_agent(
    model=model,
    tools=[search_medical_records],
    system_prompt="""
    You are a helpful medical records assistant.

    Use search_medical_records when answering questions
    about patient records.

    Base your answers on retrieved information.
    Never invent medical information.
    If the records do not contain the answer, say so clearly.
    Answer naturally and directly.
    """
)

# ----------------------------------
#  Ask the agent
# ----------------------------------

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What treatment and follow-up did patient 101 receive?"
        }
    ]
})

print(result["messages"][-1].content)


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview"
)

documents = [
    Document(
        page_content="Batman is a superhero movie.",
        metadata={"genre": "action"}
    ),
    Document(
        page_content="The Notebook is a romantic movie.",
        metadata={"genre": "romance"}
    ),
    Document(
        page_content="John Wick is an action thriller.",
        metadata={"genre": "action"}
    )
]

vectorstore:FAISS = FAISS.from_documents(
    documents,
    embeddings
)



